# PS S6E6 - 016 Blend add015 RealMLP check

Experiment: `exp_20260605_016_blend_add015_realmlp_check`

Purpose:
- Add `015 shared001 RealMLP as-is` to the current best branch.
- Compare 015 against 014 current best and earlier blend references.
- Check OOF correlation / disagreement / error overlap.
- Evaluate simple average, HC, NM, LogReg, Ridge, ElasticNet.
- Save only the best blend OOF/pred `.npy` files for the next experiment.

Current context:
- 014 bias-search-on-013-best-blend:
  - CV: `0.9660085788215015`
  - Public LB: `0.96703`
- 015 shared001 RealMLP as-is:
  - CV: `0.9681693449222352`
  - Public LB: `0.96977`

Main question:
- Is 015 strong enough alone, or does 014 still add complementary signal?
- Should next experiment be 015 bias search, 014+015 blend bias search, or both?

In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import os
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.linear_model import Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260605_016_blend_add015_realmlp_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

WORKDIR = Path("/kaggle/working")
OUTDIR = WORKDIR / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

# Only load the assets that matter for the current decision.
# 013 is kept as a reference because 014 is a postprocess of 013.
MODEL_SPECS = [
    {
        "key": "012_lgb010_bias",
        "exp_id": "exp_20260605_012_lgb010_bias_search",
        "family": "LightGBM",
        "role": "single_lgb_bias_reference",
        "oof": "oof_exp_20260605_012_lgb010_bias_search_proba.npy",
        "pred": "pred_exp_20260605_012_lgb010_bias_search_proba.npy",
        "cv": 0.9632282943283176,
        "public_lb": 0.96466,
    },
    {
        "key": "013_blend_best",
        "exp_id": "exp_20260605_013_blend_add012_lgb010_bias_check",
        "family": "Blend",
        "role": "previous_blend_reference",
        "oof": "oof_exp_20260605_013_blend_add012_lgb010_bias_check_best_blend_proba.npy",
        "pred": "pred_exp_20260605_013_blend_add012_lgb010_bias_check_best_blend_proba.npy",
        "cv": 0.9648069017992585,
        "public_lb": 0.96562,
    },
    {
        "key": "014_blend_bias",
        "exp_id": "exp_20260605_014_bias_search_on_013_best_blend",
        "family": "Blend",
        "role": "previous_current_best_blend_bias",
        "oof": "oof_exp_20260605_014_bias_search_on_013_best_blend_proba.npy",
        "pred": "pred_exp_20260605_014_bias_search_on_013_best_blend_proba.npy",
        "cv": 0.9660085788215015,
        "public_lb": 0.96703,
    },
    {
        "key": "015_realmlp_shared001",
        "exp_id": "exp_20260605_015_shared001_realmlp_as_is",
        "family": "RealMLP",
        "role": "current_best_single_realmlp",
        "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "cv": 0.9681693449222352,
        "public_lb": 0.96977,
    },
]

BLEND_SETS = {
    # Core decision sets.
    "A_015_only": ["015_realmlp_shared001"],
    "B_014_only": ["014_blend_bias"],
    "C_014_015": ["014_blend_bias", "015_realmlp_shared001"],

    # Check whether pre-bias 013 adds anything once 014 and 015 exist.
    "D_013_014_015": ["013_blend_best", "014_blend_bias", "015_realmlp_shared001"],

    # Check whether 012 adds anything beyond 014 and 015.
    "E_012_014_015": ["012_lgb010_bias", "014_blend_bias", "015_realmlp_shared001"],
    "F_012_013_014_015": ["012_lgb010_bias", "013_blend_best", "014_blend_bias", "015_realmlp_shared001"],

    # Pairs for diagnostics.
    "G_013_015": ["013_blend_best", "015_realmlp_shared001"],
    "H_012_015": ["012_lgb010_bias", "015_realmlp_shared001"],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)

EXP_ID: exp_20260605_016_blend_add015_realmlp_check
OUTDIR: /kaggle/working/exp_20260605_016_blend_add015_realmlp_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def load_npy_from_dataset(filename):
    path = NPY_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing npy file: {path}\n"
            f"Add the output file to the npy-files Kaggle dataset or edit MODEL_SPECS."
        )
    return path


def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape, n_rows, n_classes)
    assert np.isfinite(arr).all(), name
    row_sum = arr.sum(axis=1)
    assert np.allclose(row_sum, 1.0, atol=1e-4), (name, row_sum.min(), row_sum.max())
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())


def normalize_rows(p, eps=1e-12):
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)


def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)


def proba_to_pred(p):
    return np.argmax(p, axis=1)


def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)


def corr_pearson(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def corr_spearman(a, b):
    a = np.asarray(a, dtype=float).reshape(-1)
    b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)


def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out


def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res


def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)


def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats.append(p)
            mats.append(logit_transform(p))
        elif mode == "raw_rank_logit":
            mats.append(p)
            mats.append(rank_normalize_matrix(p))
            mats.append(logit_transform(p))
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)


def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg


def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    cur_score = balanced_accuracy_score(y_true, cur.argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]

    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score = cur_score
            best_w = w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w = cand_w / cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score = score
                        best_w = cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score = best_score
                w = best_w
                improved = True
                hist.append({"iter": len(hist), "score": float(cur_score), "weights": w.copy().tolist()})

    final = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    return w, final, float(cur_score), hist


def nm_optimize_weights(y_true, probas, maxiter=1500):
    n = len(probas)
    def unpack(theta):
        e = np.exp(theta - np.max(theta))
        return e / e.sum()

    def objective(theta):
        w = unpack(theta)
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return -balanced_accuracy_score(y_true, p.argmax(axis=1))

    res = minimize(objective, np.zeros(n), method="Nelder-Mead",
                   options={"maxiter": maxiter, "xatol": 1e-7, "fatol": 1e-10})
    w = unpack(res.x)
    p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
    score = balanced_accuracy_score(y_true, p.argmax(axis=1))
    return w, p, float(score), res


def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)
if not NPY_DIR.exists():
    raise FileNotFoundError(NPY_DIR)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)

assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof = {}
pred = {}
resolved_specs = []

for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = load_npy_from_dataset(spec["oof"])
    pred_path = load_npy_from_dataset(spec["pred"])

    oof_arr = np.load(oof_path).astype(np.float32)
    pred_arr = np.load(pred_path).astype(np.float32)

    validate_proba(f"{key} oof", oof_arr, len(train), n_classes)
    validate_proba(f"{key} pred", pred_arr, len(test), n_classes)

    oof[key] = oof_arr
    pred[key] = pred_arr

    row = dict(spec)
    row["resolved_oof_path"] = str(oof_path)
    row["resolved_pred_path"] = str(pred_path)
    row["oof_shape"] = list(oof_arr.shape)
    row["pred_shape"] = list(pred_arr.shape)
    resolved_specs.append(row)

model_keys = [s["key"] for s in MODEL_SPECS]

print("train:", train.shape)
print("test :", test.shape)
print("class_names:", class_names)
display(pd.DataFrame(resolved_specs))

train: (577347, 12)
test : (247435, 11)
class_names: ['GALAXY', 'QSO', 'STAR']


,key,exp_id,family,role,oof,pred,cv,public_lb,resolved_oof_path,resolved_pred_path,oof_shape,pred_shape
0,012_lgb010_bias,exp_20260605_012_lgb010_bias_search,LightGBM,single_lgb_bias_reference,oof_exp_20260605_012_lgb010_bias_search_proba.npy,pred_exp_20260605_012_lgb010_bias_search_proba...,0.963228,0.96466,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
1,013_blend_best,exp_20260605_013_blend_add012_lgb010_bias_check,Blend,previous_blend_reference,oof_exp_20260605_013_blend_add012_lgb010_bias_...,pred_exp_20260605_013_blend_add012_lgb010_bias...,0.964807,0.96562,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
2,014_blend_bias,exp_20260605_014_bias_search_on_013_best_blend,Blend,previous_current_best_blend_bias,oof_exp_20260605_014_bias_search_on_013_best_b...,pred_exp_20260605_014_bias_search_on_013_best_...,0.966009,0.96703,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
3,015_realmlp_shared001,exp_20260605_015_shared001_realmlp_as_is,RealMLP,current_best_single_realmlp,oof_exp_20260605_015_shared001_realmlp_as_is_p...,pred_exp_20260605_015_shared001_realmlp_as_is_...,0.968169,0.96977,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"


In [4]:
# ============================================================
# 3. Individual scores
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)

    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

,key,exp_id,family,role,cv_from_memo,public_lb,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
3,015_realmlp_shared001,exp_20260605_015_shared001_realmlp_as_is,RealMLP,current_best_single_realmlp,0.968169,0.96977,0.968169,0.0,0.962234,0.976098,0.966177
2,014_blend_bias,exp_20260605_014_bias_search_on_013_best_blend,Blend,previous_current_best_blend_bias,0.966009,0.96703,0.966009,0.0,0.968062,0.970617,0.959347
1,013_blend_best,exp_20260605_013_blend_add012_lgb010_bias_check,Blend,previous_blend_reference,0.964807,0.96562,0.964807,0.0,0.971765,0.968833,0.953822
0,012_lgb010_bias,exp_20260605_012_lgb010_bias_search,LightGBM,single_lgb_bias_reference,0.963228,0.96466,0.963228,0.0,0.964634,0.968944,0.956107


In [5]:
# ============================================================
# 4. Pairwise OOF correlation / disagreement / error overlap
# ============================================================

pair_rows = []
pred_labels = {k: proba_to_pred(oof[k]) for k in model_keys}
wrong_flags = {k: pred_labels[k] != y for k in model_keys}

for a, b in combinations(model_keys, 2):
    pa, pb = oof[a], oof[b]
    ya, yb = pred_labels[a], pred_labels[b]
    wrong_a, wrong_b = wrong_flags[a], wrong_flags[b]
    both = wrong_a & wrong_b
    either = wrong_a | wrong_b

    row = {
        "model_a": a,
        "model_b": b,
        "pearson_flat_proba": corr_pearson(flatten_proba(pa), flatten_proba(pb)),
        "spearman_flat_proba": corr_spearman(flatten_proba(pa), flatten_proba(pb)),
        "argmax_agreement": float((ya == yb).mean()),
        "argmax_disagreement": float((ya != yb).mean()),
        "wrong_rate_a": float(wrong_a.mean()),
        "wrong_rate_b": float(wrong_b.mean()),
        "both_wrong_rate": float(both.mean()),
        "either_wrong_rate": float(either.mean()),
        "error_jaccard": float(both.sum() / max(either.sum(), 1)),
        "a_wrong_b_correct_rate": float((wrong_a & ~wrong_b).mean()),
        "a_correct_b_wrong_rate": float((~wrong_a & wrong_b).mean()),
    }
    for ci, cls in enumerate(class_names):
        row[f"pearson_proba_{cls}"] = corr_pearson(pa[:, ci], pb[:, ci])
        row[f"spearman_proba_{cls}"] = corr_spearman(pa[:, ci], pb[:, ci])
    pair_rows.append(row)

pairwise_df = pd.DataFrame(pair_rows).sort_values("pearson_flat_proba")
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

,model_a,model_b,pearson_flat_proba,spearman_flat_proba,argmax_agreement,argmax_disagreement,wrong_rate_a,wrong_rate_b,both_wrong_rate,either_wrong_rate,error_jaccard,a_wrong_b_correct_rate,a_correct_b_wrong_rate,pearson_proba_GALAXY,spearman_proba_GALAXY,pearson_proba_QSO,spearman_proba_QSO,pearson_proba_STAR,spearman_proba_STAR
2,012_lgb010_bias,015_realmlp_shared001,0.991665,0.912138,0.983012,0.016988,0.035713,0.034388,0.026748,0.043353,0.616980,0.008965,0.007640,0.989504,0.932024,0.993429,0.825957,0.983870,0.769173
4,013_blend_best,015_realmlp_shared001,0.993310,0.913463,0.985011,0.014989,0.031401,0.034388,0.025550,0.040239,0.634943,0.005851,0.008839,0.991536,0.941769,0.995401,0.851532,0.987408,0.771536
5,014_blend_bias,015_realmlp_shared001,0.994167,0.916308,0.985937,0.014063,0.032668,0.034388,0.026648,0.040409,0.659451,0.006021,0.007741,0.992597,0.941779,0.995879,0.854700,0.988644,0.771301
0,012_lgb010_bias,013_blend_best,0.997424,0.991759,0.991374,0.008626,0.035713,0.031401,0.029308,0.037806,0.775233,0.006405,0.002092,0.996656,0.990642,0.998528,0.969219,0.995030,0.987956
1,012_lgb010_bias,014_blend_bias,0.997970,0.992720,0.992885,0.007115,0.035713,0.032668,0.030697,0.037684,0.814588,0.005016,0.001971,0.997331,0.990643,0.998812,0.969905,0.995822,0.989834
3,013_blend_best,014_blend_bias,0.999769,0.999386,0.996401,0.003599,0.031401,0.032668,0.030247,0.033822,0.894300,0.001154,0.002421,0.999709,1.000000,0.999861,0.999292,0.999654,0.999256


In [6]:
# ============================================================
# 5. Correlation matrices
# ============================================================

pearson_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
spearman_flat = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
argmax_agreement = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)
error_jaccard = pd.DataFrame(index=model_keys, columns=model_keys, dtype=float)

for a in model_keys:
    for b in model_keys:
        pearson_flat.loc[a, b] = corr_pearson(flatten_proba(oof[a]), flatten_proba(oof[b]))
        spearman_flat.loc[a, b] = corr_spearman(flatten_proba(oof[a]), flatten_proba(oof[b]))
        argmax_agreement.loc[a, b] = float((pred_labels[a] == pred_labels[b]).mean())
        both = wrong_flags[a] & wrong_flags[b]
        either = wrong_flags[a] | wrong_flags[b]
        error_jaccard.loc[a, b] = float(both.sum() / max(either.sum(), 1))

display(pearson_flat)
display(spearman_flat)
display(argmax_agreement)
display(error_jaccard)

pearson_flat.to_csv(OUTDIR / "corr_matrix_pearson_flat_proba.csv")
spearman_flat.to_csv(OUTDIR / "corr_matrix_spearman_flat_proba.csv")
argmax_agreement.to_csv(OUTDIR / "matrix_argmax_agreement.csv")
error_jaccard.to_csv(OUTDIR / "matrix_error_jaccard.csv")

,012_lgb010_bias,013_blend_best,014_blend_bias,015_realmlp_shared001
012_lgb010_bias,1.000000,0.997424,0.997970,0.991665
013_blend_best,0.997424,1.000000,0.999769,0.993310
014_blend_bias,0.997970,0.999769,1.000000,0.994167
015_realmlp_shared001,0.991665,0.993310,0.994167,1.000000


,012_lgb010_bias,013_blend_best,014_blend_bias,015_realmlp_shared001
012_lgb010_bias,1.000000,0.991759,0.992720,0.912138
013_blend_best,0.991759,1.000000,0.999386,0.913463
014_blend_bias,0.992720,0.999386,1.000000,0.916308
015_realmlp_shared001,0.912138,0.913463,0.916308,1.000000


,012_lgb010_bias,013_blend_best,014_blend_bias,015_realmlp_shared001
012_lgb010_bias,1.000000,0.991374,0.992885,0.983012
013_blend_best,0.991374,1.000000,0.996401,0.985011
014_blend_bias,0.992885,0.996401,1.000000,0.985937
015_realmlp_shared001,0.983012,0.985011,0.985937,1.000000


,012_lgb010_bias,013_blend_best,014_blend_bias,015_realmlp_shared001
012_lgb010_bias,1.000000,0.775233,0.814588,0.616980
013_blend_best,0.775233,1.000000,0.894300,0.634943
014_blend_bias,0.814588,0.894300,1.000000,0.659451
015_realmlp_shared001,0.616980,0.634943,0.659451,1.000000


In [7]:
# ============================================================
# 6. Blend diagnostics
# ============================================================

blend_rows = {}

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row


for set_name, keys in BLEND_SETS.items():
    print(f"===== {set_name}: {keys} =====")

    probas = [oof[k] for k in keys]
    pred_probas = [pred[k] for k in keys]

    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred, weights=np.ones(len(keys)) / len(keys))

    if len(keys) >= 2:
        hc_w, hc_oof, hc_score, hc_hist = hill_climb_weights(y, probas)
        hc_pred = normalize_rows(sum(hc_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=hc_w, extra={"hc_history_len": len(hc_hist)})

        nm_w, nm_oof, nm_score, nm_res = nm_optimize_weights(y, probas)
        nm_pred = normalize_rows(sum(nm_w[i] * pred_probas[i] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=nm_w, extra={"nm_success": bool(nm_res.success), "nm_message": str(nm_res.message)})

    # Linear meta is in-sample on OOF predictions.
    # Treat improvements as screening only, not final nested OOF.
    for mode in ["raw", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)

        try:
            lr = LogisticRegression(C=0.05, penalty="l2", solver="lbfgs", max_iter=2000, random_state=SEED, n_jobs=-1)
            lr.fit(X_meta, y)
            lr_oof = lr.predict_proba(X_meta)
            lr_pred = lr.predict_proba(X_test_meta)
            record_blend(set_name, f"logreg_{mode}", keys, lr_oof, lr_pred, extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")

        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            rc_oof = softmax_np(rc.decision_function(X_meta), axis=1)
            rc_pred = softmax_np(rc.decision_function(X_test_meta), axis=1)
            record_blend(set_name, f"ridgeclf_{mode}", keys, rc_oof, rc_pred, extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")

        # Regression-style Ridge / ElasticNet to one-hot targets.
        # These are also in-sample screening only.
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            rr_oof = normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None))
            rr_pred = normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None))
            record_blend(set_name, f"ridge_reg_{mode}", keys, rr_oof, rr_pred, extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")

        try:
            y_oh = onehot_y(y, n_classes)
            en_preds_oof = []
            en_preds_test = []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                en_preds_oof.append(en.predict(X_meta))
                en_preds_test.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(en_preds_oof).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(en_preds_test).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")


blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False)
display(blend_df.head(120))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

===== A_015_only: ['015_realmlp_shared001'] =====
[WARN] logreg failed: A_015_only logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_015_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] ridge_reg failed: A_015_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] elasticnet_reg failed: A_015_only logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] logreg failed: A_015_only raw_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_015_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] ridge_reg failed: A_015_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] elasticnet_reg failed: A_015_only raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] logreg failed: A_015_only raw

,set_name,method,keys,n_models,balanced_accuracy,weights_json,recall_GALAXY,recall_QSO,recall_STAR,C,risk,alpha,l1_ratio,hc_history_len,nm_success,nm_message
51,G_013_015,hc_nonnegative_raw,"013_blend_best,015_realmlp_shared001",2,0.968247,"{""013_blend_best"": 0.1520544869757452, ""015_re...",0.964056,0.975329,0.965355,NaN,NaN,NaN,NaN,11.0,NaN,NaN
23,C_014_015,hc_nonnegative_raw,"014_blend_bias,015_realmlp_shared001",2,0.968244,"{""014_blend_bias"": 0.3927248436266472, ""015_re...",0.965503,0.974441,0.964787,NaN,NaN,NaN,NaN,4.0,NaN,NaN
37,E_012_014_015,hc_nonnegative_raw,"012_lgb010_bias,014_blend_bias,015_realmlp_sha...",3,0.968244,"{""012_lgb010_bias"": 0.0, ""014_blend_bias"": 0.3...",0.965503,0.974441,0.964787,NaN,NaN,NaN,NaN,8.0,NaN,NaN
30,D_013_014_015,hc_nonnegative_raw,"013_blend_best,014_blend_bias,015_realmlp_shar...",3,0.968232,"{""013_blend_best"": 0.0, ""014_blend_bias"": 0.34...",0.965055,0.974638,0.965004,NaN,NaN,NaN,NaN,10.0,NaN,NaN
44,F_012_013_014_015,hc_nonnegative_raw,"012_lgb010_bias,013_blend_best,014_blend_bias,...",4,0.968214,"{""012_lgb010_bias"": 0.04447992508889849, ""013_...",0.965421,0.974339,0.964883,NaN,NaN,NaN,NaN,11.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14,B_014_only,logreg_raw_logit,014_blend_bias,1,0.960148,None,0.978873,0.965205,0.936367,0.05,in_sample_meta_screening,NaN,NaN,NaN,NaN,NaN
18,B_014_only,logreg_raw_rank_logit,014_blend_bias,1,0.960082,None,0.978921,0.965188,0.936137,0.05,in_sample_meta_screening,NaN,NaN,NaN,NaN,NaN
13,B_014_only,elasticnet_reg_logit,014_blend_bias,1,0.952942,None,0.979562,0.950163,0.929102,NaN,in_sample_meta_screening,0.0005,0.05,NaN,NaN,NaN
11,B_014_only,ridgeclf_logit,014_blend_bias,1,0.952910,None,0.979564,0.950138,0.929029,NaN,in_sample_meta_screening,10.0000,NaN,NaN,NaN,NaN


In [8]:
# ============================================================
# 7. Save best non-meta blend only
# ============================================================

# For a fair next experiment, save only the best non-meta proba blend.
# LogReg/Ridge/ElasticNet rows above are in-sample meta screening and are not nested OOF.
# They should not be used as final OOF inputs unless a proper nested meta notebook is built.

safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy()
safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)

best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
elif best_method in ["hc_nonnegative_raw", "nm_softmax_raw"]:
    probas = [oof[k] for k in best_keys]
    pred_probas = [pred[k] for k in best_keys]
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * probas[i] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred_probas[i] for i in range(len(best_keys))))
else:
    raise RuntimeError(best_method)

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / "oof_exp_20260605_016_blend_add015_realmlp_check_best_blend_proba.npy"
best_pred_npy = OUTDIR / "pred_exp_20260605_016_blend_add015_realmlp_check_best_blend_proba.npy"

np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET_COL: best_labels,
})

assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / "submission_exp_20260605_016_blend_add015_realmlp_check_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}

save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_df = pd.DataFrame([{
    "set_name": best_set_name,
    "method": best_method,
    "balanced_accuracy": float(best_oof_score),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)

Best safe blend:
{
  "set_name": "G_013_015",
  "method": "hc_nonnegative_raw",
  "keys": "013_blend_best,015_realmlp_shared001",
  "n_models": 2,
  "balanced_accuracy": 0.968246781866073,
  "weights_json": "{\"013_blend_best\": 0.1520544869757452, \"015_realmlp_shared001\": 0.8479455130242548}",
  "recall_GALAXY": 0.9640563738476211,
  "recall_QSO": 0.9753292983789044,
  "recall_STAR": 0.9653546733716938,
  "C": NaN,
  "risk": NaN,
  "alpha": NaN,
  "l1_ratio": NaN,
  "hc_history_len": 11.0,
  "nm_success": NaN,
  "nm_message": NaN
}
best_oof_score: 0.968246781866073


,set_name,method,balanced_accuracy,submission,oof_proba,pred_proba
0,G_013_015,hc_nonnegative_raw,0.968247,submission_exp_20260605_016_blend_add015_realm...,oof_exp_20260605_016_blend_add015_realmlp_chec...,pred_exp_20260605_016_blend_add015_realmlp_che...


In [9]:
# ============================================================
# 8. Role judgement
# ============================================================

def score_of(key):
    return float(individual_df.loc[individual_df["key"] == key, "recomputed_oof_cv"].iloc[0])

judgement = {
    "reference_scores": {
        "014_cv": 0.9660085788215015,
        "014_public_lb": 0.96703,
        "015_cv": 0.9681693449222352,
        "015_public_lb": 0.96977,
    },
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "main_questions": {
        "does_014_add_to_015": "Check C_014_015 safe methods and weights.",
        "does_013_add_after_014_015": "Check D_013_014_015.",
        "does_012_add_after_014_015": "Check E/F.",
        "next_bias_target": "If best safe blend beats 015, run bias search on 016 best. Otherwise run bias search on 015 single.",
    },
    "initial_policy": {
        "015_realmlp_shared001": "current_best_single_reference",
        "014_blend_bias": "previous_best_blend_bias_reference",
        "016_best_safe_blend": "candidate_for_submission_and_next_bias_search_if_it_beats_015",
        "linear_meta_rows": "screening_only_not_final_without_nested_meta",
    },
}

save_json(judgement, OUTDIR / "role_judgement.json")
print(json.dumps(judgement, indent=2, ensure_ascii=False))

{
  "reference_scores": {
    "014_cv": 0.9660085788215015,
    "014_public_lb": 0.96703,
    "015_cv": 0.9681693449222352,
    "015_public_lb": 0.96977
  },
  "individual_best": {
    "key": "015_realmlp_shared001",
    "cv": 0.9681693449222352,
    "public_lb": 0.96977
  },
  "best_safe_blend": {
    "exp_id": "exp_20260605_016_blend_add015_realmlp_check",
    "best_set_name": "G_013_015",
    "best_method": "hc_nonnegative_raw",
    "best_keys": [
      "013_blend_best",
      "015_realmlp_shared001"
    ],
    "cv_score": 0.968246781866073,
    "weights_json": "{\"013_blend_best\": 0.1520544869757452, \"015_realmlp_shared001\": 0.8479455130242548}",
    "oof_path": "/kaggle/working/exp_20260605_016_blend_add015_realmlp_check/oof_exp_20260605_016_blend_add015_realmlp_check_best_blend_proba.npy",
    "pred_path": "/kaggle/working/exp_20260605_016_blend_add015_realmlp_check/pred_exp_20260605_016_blend_add015_realmlp_check_best_blend_proba.npy",
    "submission_path": "/kaggle/working/

In [10]:
# ============================================================
# 9. Save summary / memo
# ============================================================

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Add 015 RealMLP to current 014 branch and evaluate correlation/blend",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "blend_add015_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 015 shared001 RealMLP",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-05",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from the fixed npy-files Kaggle dataset folder, "
            "add 015 shared001 RealMLP as-is to the current 014 branch, "
            "and decide whether 014 adds complementary signal to 015. "
            "Save only the best safe blend OOF/pred proba for the next experiment."
        ),
        "success_criteria": [
            "load 012/013/014/015 OOF and pred npy files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations",
            "evaluate predefined blend sets centered on 014/015",
            "separate safe blends from in-sample meta screening",
            "save only best safe blend OOF/pred npy",
            "save best safe blend submission",
            "save memo and summary",
        ],
    },
    "inputs": {
        "npy_dir": str(NPY_DIR),
        "models": resolved_specs,
        "blend_sets": BLEND_SETS,
    },
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "corr_matrix_pearson_flat_proba": "corr_matrix_pearson_flat_proba.csv",
        "corr_matrix_spearman_flat_proba": "corr_matrix_spearman_flat_proba.csv",
        "matrix_argmax_agreement": "matrix_argmax_agreement.csv",
        "matrix_error_jaccard": "matrix_error_jaccard.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "best_blend_info": "best_blend_info.json",
        "saved_submission_candidates": "saved_submission_candidates.csv",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "role_judgement": "role_judgement.json",
        "summary": "blend_add015_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "015 is currently stronger than 014 as a single model. "
            "The key question is whether 014 contributes enough complementary error structure to improve a safe blend. "
            "In-sample meta rows are diagnostic only."
        ),
        "next_action": [
            "Review individual_scores.csv",
            "Review pairwise_oof_correlation.csv, especially 014 vs 015",
            "Review safe rows in blend_diagnostics.csv",
            "Submit best safe blend if it improves over 015 enough",
            "If best safe blend beats 015, run bias search on 016 best blend",
            "If not, run bias search on 015 single RealMLP",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)

Saved files:
 - best_blend_info.json
 - blend_add015_summary.json
 - blend_diagnostics.csv
 - corr_matrix_pearson_flat_proba.csv
 - corr_matrix_spearman_flat_proba.csv
 - individual_scores.csv
 - matrix_argmax_agreement.csv
 - matrix_error_jaccard.csv
 - memo.yml
 - oof_exp_20260605_016_blend_add015_realmlp_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pred_exp_20260605_016_blend_add015_realmlp_check_best_blend_proba.npy
 - role_judgement.json
 - saved_submission_candidates.csv
 - submission_exp_20260605_016_blend_add015_realmlp_check_best_blend.csv
